In [20]:
import sys
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project")
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project\src")

from functions_v2 import *
from two_level_mc import *

In [6]:
edges, n_vertices, weights = load_graph(r"../../data/raw/power-US-Grid.mtx")
print(f"n_vertices={n_vertices}, edges={len(edges)}")

✓ Loaded: ../../data/raw/power-US-Grid.mtx
  Vertices : 4941
  Edges    : 6594
  Weighted : no

n_vertices=4941, edges=6594


In [9]:
# 3. Phase 1
A, D, L = build_graph_matrices(edges, n_vertices)

# 4. Phase 2 — sparse from the start, given graph size
lambda_min = compute_lambda_min(L, D)
L_sigma = build_shifted_laplacian(L, D, lambda_min)
print("L_sigma type:", type(L_sigma))

✓ Phase 1 complete: A, D, L built as sparse matrices
  Matrix size : 4941 x 4941
  Degree range: [1, 19]
  Non-zeros in L: 18129

✓ lambda_min = 0.000271
  (eigenvalues found: [0.         0.00027102])
✓ Phase 2 complete: L_sigma built (sigma^2 = 0.25)
  L_sigma type: sparse

L_sigma type: <class 'scipy.sparse._csc.csc_matrix'>


In [11]:
G_nx = nx.Graph()
G_nx.add_edges_from(edges)

In [13]:
gamma_in, gamma_out, diameter = find_diameter_endpoints(G_nx, n_sample=5, k_hop=4)

In [15]:
diameter

46

In [22]:
check_boundary_fraction(gamma_in, gamma_out, n_vertices)

gamma_in: 23, gamma_out: 14, n_vertices: 4941
Boundary fraction: 0.7488%
  -> Likely safe for aggregation (comparable to validated successes).


0.00748836267961951

In [27]:
aggregate_of, n_coarse = build_grouped_aggregation(G_nx, gamma_in, gamma_out, max_size=10)

# ---- Step 1: check the aggregation is actually usable ----
summary = summarize_aggregation(aggregate_of, n_coarse, gamma_in, gamma_out)

n_coarse: 2056 (2056 aggregates)
Size distribution -- min: 1, max: 10, mean: 2.40
Singletons: 7 (0.3%)
gamma_in_coarse: 11, gamma_out_coarse: 4
Overlap (must be empty): set()
Interior coarse vertices: 2041 (99.3%)


In [30]:
# ---- Step 2: build the coarse graph edges ----
coarse_edges, coarse_contribs = build_coarse_graph_edges(edges, aggregate_of)
print(f"coarse edges: {len(coarse_edges)} (from {len(edges)} fine edges)")

# ---- Step 3: build the fine-to-coarse boundary mapping ----
gamma_in_coarse = sorted(set(aggregate_of[v] for v in gamma_in))
gamma_out_coarse = sorted(set(aggregate_of[v] for v in gamma_out))
overlap = set(gamma_in_coarse) & set(gamma_out_coarse)
print(f"gamma_in_coarse: {len(gamma_in_coarse)}, gamma_out_coarse: {len(gamma_out_coarse)}")
print(f"Overlap (must be empty): {overlap}")

coarse edges: 3260 (from 6594 fine edges)
gamma_in_coarse: 11, gamma_out_coarse: 4
Overlap (must be empty): set()


In [33]:
from sksparse.cholmod import cholesky as sparse_cholesky

setup = TwoLevelSetup.build(
    edges, n_vertices, L_sigma, lambda_min,
    gamma_in, gamma_out,
    build_incidence_matrix, sparse_cholesky,
    max_size=10
)

gamma_in: 23, gamma_out: 14, n_vertices: 4941
Boundary fraction: 0.7488%
  -> Likely safe for aggregation (comparable to validated successes).
n_coarse: 2074 (2074 aggregates)
Size distribution -- min: 2, max: 10, mean: 2.38
Singletons: 0 (0.0%)
gamma_in_coarse: 15, gamma_out_coarse: 7
Overlap (must be empty): set()
Interior coarse vertices: 2052 (98.9%)
coarse edges: 3282 (from 6594 fine edges)


In [35]:
import time

t0 = time.time()
Q_samples = monte_carlo_loop_tqdm(L_sigma, lambda_min, edges, n_vertices,
                                    gamma_in, gamma_out, N=1000, debug=False)
single_level_time = time.time() - t0

single_level_se = Q_samples.std() / np.sqrt(len(Q_samples))

print(f"\n=== Single-Level MC on US Power Grid ===")
print(f"Time: {single_level_time:.2f}s")
print(f"Mean Q: {Q_samples.mean():.6f}")
print(f"Std Q: {Q_samples.std():.6f}")
print(f"Standard error: {single_level_se:.6f}")

Monte Carlo: 100%|██████████| 1000/1000 [00:42<00:00, 23.50sample/s, mean Q=22.3293]


✓ Done — 1000 samples
  Mean Q : 22.329339
  Std Q  : 545.881476

=== Single-Level MC on US Power Grid ===
Time: 42.72s
Mean Q: 22.329339
Std Q: 545.881476
Standard error: 17.262288


In [52]:
import time

t0 = time.time()
result = run_paired_validation(setup, N=300)
paired_time = time.time() - t0
print(f"Paired time (N=100): {paired_time:.2f}s")

Paired samples: 100%|██████████| 300/300 [00:20<00:00, 14.83sample/s, Q_fine=7.2139, Q_coarse=11.2657] 


N = 300 paired samples
Q_fine   : mean=7.213877  var=2385.803171
Q_coarse : mean=11.265725  var=5855.292466
Q_fine - Q_coarse : mean=-4.051848  var=765.990641
Correlation(Q_fine, Q_coarse): 1.0000
Variance reduction: 3.11x
Paired time (N=100): 20.23s
